## 🔧 3. Agents with Built in tools

**Goal**: Enhance agents with built-in tools — **Code Interpreter**, **File Search**, **Web Search**, and custom **Function Tools** — registered server-side in Foundry with full **tracing**, then used via `FoundryAgent`.

- `CodeInterpreterTool` — data analysis & charts (server-side)
- `FileSearchTool` — RAG over uploaded documents (server-side)
- `WebSearchTool` — real-time web search (server-side)
- `FunctionTool` + `@tool` decorator — custom Python functions
- **Tracing** — all interactions visible in Foundry portal

In [1]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import sys, os
from pathlib import Path
sys.path.insert(0, os.path.abspath("..") if "notebooks" in os.getcwd() else ".")

from dotenv import load_dotenv
load_dotenv()

from utils.config import get_env, get_credential
from agent_framework import tool

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    PromptAgentDefinition,
    CodeInterpreterTool,
    FileSearchTool,
    WebSearchTool,
    FunctionTool as AzureFunctionTool,
)
from agent_framework.foundry import FoundryAgent

# Endpoints
PROJECT_ENDPOINT = get_env("AZURE_AI_PROJECT_ENDPOINT_FULL")
MODEL = get_env("AZURE_AI_MODEL_DEPLOYMENT_NAME")

project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=get_credential(),
)

print(f"📡 Project endpoint: {PROJECT_ENDPOINT}")
print(f"🧠 Model: {MODEL}")
print("✅ Connected to Foundry")

c:\Dibyendu\Code\agent-framework-demo-py\.venv\Lib\site-packages\agent_framework\_skills.py:116: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
c:\Dibyendu\Code\agent-framework-demo-py\.venv\Lib\site-packages\agent_framework\_harness\_memory.py:651: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.


📡 Project endpoint: https://foundry-demo-us-east-resource.services.ai.azure.com/api/projects/prj-foundry-demo-us-east-dev-001
🧠 Model: gpt-4o
✅ Connected to Foundry


---
### Part A — Code Interpreter 🧮
Register an agent with `CodeInterpreterTool` in Foundry, then use it via `FoundryAgent`.
The agent writes and executes Python code to analyze data and create charts — all traced server-side.

In [2]:
# ── A1. Register Code Interpreter agent in Foundry ───────────────────────────
CODE_AGENT_NAME = "DataAnalyst-af"

code_agent_version = project.agents.create_version(
    agent_name=CODE_AGENT_NAME,
    definition=PromptAgentDefinition(
        model=MODEL,
        instructions=(
            "You are a data analyst. When given data, write and execute Python "
            "code to analyze it. Always produce a chart when asked for visualization. "
            "Explain your findings clearly."
        ),
        tools=[CodeInterpreterTool()],
    ),
)

# Connect via Agent Framework
code_agent = FoundryAgent(
    project_endpoint=PROJECT_ENDPOINT,
    agent_name=code_agent_version.name,
    agent_version=str(code_agent_version.version),
    credential=get_credential(),
)

print(f"✅ Agent registered in Foundry: {code_agent_version.name} v{code_agent_version.version}")
print(f"🤖 FoundryAgent connected: {type(code_agent).__name__}")

2026-05-28 16:31:24,151 - azure.ai.agentserver - INFO - Application Insights not configured, telemetry export disabled.
✅ Agent registered in Foundry: DataAnalyst-af v1
🤖 FoundryAgent connected: FoundryAgent


In [3]:
# ── A2. Ask for analysis with code execution ─────────────────────────────────
from agent_framework import Content

result = await code_agent.run(
    "Calculate the first 20 Fibonacci numbers and create a line chart showing their growth."
)
print(f"🗨️  Agent:\n{result}")

# Inspect generated code
for message in result.messages:
    code_blocks = [c for c in message.contents if c.type == "code_interpreter_tool_call"]
    outputs = [c for c in message.contents if c.type == "code_interpreter_tool_result"]
    if code_blocks:
        for content in (code_blocks[0].inputs or []):
            if isinstance(content, Content) and content.type == "text":
                print(f"\n📝 Generated code:\n{content.text}")
                break
    if outputs:
        for out in (outputs[0].outputs or []):
            if isinstance(out, Content) and out.type == "text":
                print(f"\n📊 Execution output:\n{out.text}")

print(f"\n🔍 Check Foundry portal → Tracing to see this interaction!")

🗨️  Agent:
It seems there's an issue with the plotting backend. Let's try a different approach to create the plot. I have successfully generated the plot. You can [download the Fibonacci plot here](sandbox:/mnt/data/fibonacci_plot.png).

The line chart illustrates the growth of the first 20 Fibonacci numbers. As you can see, the sequence starts slowly but increases rapidly, showcasing the characteristic exponential growth pattern.

📝 Generated code:
import matplotlib.pyplot as plt

# Function to calculate Fibonacci numbers
def fibonacci_sequence(n):
    fib_sequence = [0, 1]
    for i in range(2, n):
        next_number = fib_sequence[i-1] + fib_sequence[i-2]
        fib_sequence.append(next_number)
    return fib_sequence

# Calculate the first 20 Fibonacci numbers
fibonacci_numbers = fibonacci_sequence(20)

# Plotting the Fibonacci numbers
plt.figure(figsize=(10, 6))
plt.plot(range(1, 21), fibonacci_numbers, marker='o')
plt.title("First 20 Fibonacci Numbers")
plt.xlabel("Position in 

---
### Part B — File Search (RAG) 📂
Uploaded product docs into a Vector Store, register an agent with `FileSearchTool` in Foundry, then query via `FoundryAgent`.

In [ ]:
# ── B2. Register File Search agent in Foundry ────────────────────────────────
FILE_AGENT_NAME = "ProductSpecialist-af"
vectorstoredid = "vs_ZK0XW6cotxNUoZ8rBAqWShcW"

file_agent_version = project.agents.create_version(
    agent_name=FILE_AGENT_NAME,
    definition=PromptAgentDefinition(
        model=MODEL,
        instructions=(
            "You are a Contoso product specialist. Use the file search tool to "
            "look up product details, pricing, and policies from the company "
            "knowledge base. Always cite the source."
        ),
        tools=[FileSearchTool(vector_store_ids=[vectorstoredid])],
    ),
)

# Connect via Agent Framework
file_agent = FoundryAgent(
    project_endpoint=PROJECT_ENDPOINT,
    agent_name=file_agent_version.name,
    agent_version=str(file_agent_version.version),
    credential=get_credential(),
)

print(f"✅ Agent registered in Foundry: {file_agent_version.name} v{file_agent_version.version}")
print(f"🤖 FoundryAgent connected: {type(file_agent).__name__}")

✅ Agent registered in Foundry: ProductSpecialist-af v1
🤖 FoundryAgent connected: FoundryAgent


In [5]:
# ── B3. Ask questions grounded in the uploaded doc ───────────────────────────
for question in [
    "What is the price of the Contoso Widget Pro?",
    "Compare Widget Pro vs Widget Lite.",
    "What is the return policy for hardware?",
]:
    resp = await file_agent.run(question)
    print(f"\n❓ {question}")
    print(f"🗨️  {resp}")

print(f"\n🔍 Check Foundry portal → Tracing to see these interactions!")


❓ What is the price of the Contoso Widget Pro?
🗨️  The price of the Contoso Widget Pro is $49.99.

❓ Compare Widget Pro vs Widget Lite.
🗨️  ### Widget Pro vs Widget Lite Comparison

**Contoso Widget Pro:**
- **Price:** $49.99
- **SKU:** CW-PRO-001
- **Description:** A premium widget with titanium-reinforced housing and smart-connect technology, ideal for enterprise deployments.
- **Warranty:** 3 years
- **Weight:** 250g

**Contoso Widget Lite:**
- **Price:** $19.99
- **SKU:** CW-LITE-002
- **Description:** A lightweight, budget-friendly widget for personal and small-team use, easy to set up in under 5 minutes.
- **Warranty:** 1 year
- **Weight:** 120g.

The Widget Pro is designed for more demanding environments or enterprise use, featuring durable construction and advanced connectivity features. Widget Lite is more suited for personal or small-team applications, with a focus on cost-effectiveness and ease of use.

❓ What is the return policy for hardware?
🗨️  The return policy for har

---
### Part C — Web Search 🌐
Register an agent with `WebSearchTool` in Foundry to fetch real-time info from the web, all traced server-side.

In [6]:
# ── C1. Register Web Search agent in Foundry ─────────────────────────────────
WEB_AGENT_NAME = "ResearchAssistant-af"

web_agent_version = project.agents.create_version(
    agent_name=WEB_AGENT_NAME,
    definition=PromptAgentDefinition(
        model=MODEL,
        instructions=(
            "You are a research assistant. Use web search to find the latest "
            "information and always include source URLs in your response."
        ),
        tools=[WebSearchTool()],
    ),
)

# Connect via Agent Framework
web_agent = FoundryAgent(
    project_endpoint=PROJECT_ENDPOINT,
    agent_name=web_agent_version.name,
    agent_version=str(web_agent_version.version),
    credential=get_credential(),
)

print(f"✅ Agent registered in Foundry: {web_agent_version.name} v{web_agent_version.version}")
print(f"🤖 FoundryAgent connected: {type(web_agent).__name__}")

result = await web_agent.run(
    "What are the top AI announcements from Microsoft Build this year?"
)
print(f"\n🗨️  Agent: {result}")
print(f"\n🔍 Check Foundry portal → Tracing to see this interaction!")

✅ Agent registered in Foundry: ResearchAssistant-af v1
🤖 FoundryAgent connected: FoundryAgent

🗨️  Agent: Below is a detailed and well‑structured overview of the **top AI announcements and themes** from **Microsoft Build 2026**, based on the most up‑to‑date sources. All statements are thoroughly cited and annotated with their origin.

---

##  Overview: When and Where Build 2026 Took Place

- **Dates**: June 2–3, 2026, in San Francisco, California—marking the first time in nearly a decade Microsoft Build moved away from its traditional Seattle venue ([build.microsoft.com](https://build.microsoft.com/en-US/home)).
- **Format & Focus**: A deliberately more intimate, interactive event hosting around 2,500 in-person developers, complemented by free online live streams. Sessions emphasized a “no‑fluff,” deep technical dive into AI, cloud, and developer tools ([moneycontrol.com](https://www.moneycontrol.com/technology/microsoft-build-2026-announced-check-dates-expected-announcements-and-more

---
### Cleanup 🧹

In [7]:
# ── Cleanup ───────────────────────────────────────────────────────────────────
# Delete agents from Foundry
for name in [CODE_AGENT_NAME, FILE_AGENT_NAME, WEB_AGENT_NAME]:
    try:
        project.agents.delete(agent_name=name)
        print(f"🧹 Deleted agent: {name}")
    except Exception as e:
        print(f"⚠️  Could not delete {name}: {e}")

# Delete vector store and file
try:
    oai.vector_stores.delete(vector_store.id)
    oai.files.delete(file.id)
    print("🧹 Vector store and file deleted")
except Exception:
    pass

print("✅ Cleanup complete")

🧹 Deleted agent: DataAnalyst-af
🧹 Deleted agent: ProductSpecialist-af
🧹 Deleted agent: ResearchAssistant-af
✅ Cleanup complete


---
### ✅ What you demonstrated
| Capability | Foundry Definition | Agent Framework Feature |
|---|---|---|
| Data analysis & charts | `CodeInterpreterTool()` on `PromptAgentDefinition` | Runs server-side via `FoundryAgent` |
| Document Q&A (RAG) | `FileSearchTool(vector_store_ids=[...])` | Runs server-side via `FoundryAgent` |
| Real-time web knowledge | `WebSearchTool()` | Runs server-side via `FoundryAgent` |
| Server-side tracing | All interactions visible in Foundry portal → Tracing | |

**Next →** [04_agent_tools_mcp.ipynb](./04_agent_tools_mcp.ipynb) — Agent with MCP tools.